# 02 — Exploratory Data Analysis

**Project:** Where Should Amazon Air Build Its Next Hub?
**Author:** Aikerim Imanbayeva

## Purpose

Before building the Hub Fitness Index, we explore the data to:
1. Confirm the cargo-flow landscape (who's where, how big)
2. Spot geographic patterns that hint at gap-filling opportunities
3. Identify the relationship between airport capacity and current activity
4. Show where Amazon Air already has presence vs. where they don't
5. Build the visual intuition that informs scoring-weight choices

## Sections

1. National cargo overview
2. Year-over-year cargo trends (2020–2025)
3. Geographic distribution (states, regions)
4. Airport capacity vs. activity
5. Population catchment preview
6. Amazon Air coverage gap analysis
7. Key findings

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path('..').resolve()
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
RAW = PROJECT_ROOT / 'data' / 'raw'

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Brand-ish color palette (Amazon-air orange + neutral grays)
AMAZON_ORANGE = '#FF9900'
NEUTRAL = '#4A4A4A'
MUTED = '#BBBBBB'
ACCENT = '#007EB9'

print('Setup complete')

In [ ]:
# Load processed outputs from notebook 01
candidates = pd.read_csv(PROCESSED / 'candidate_airports.csv')
cargo_2025 = pd.read_csv(PROCESSED / 'cargo_summary_2025.csv')
cargo_yearly = pd.read_csv(PROCESSED / 'cargo_summary_yearly.csv')
top80 = pd.read_csv(PROCESSED / 'top_cargo_airports.csv')
counties = pd.read_csv(PROCESSED / 'county_population_with_centroids.csv')
amazon = pd.read_csv(RAW / 'amazon_air' / 'amazon_air_facilities.csv')

print(f'candidates:    {len(candidates):,} rows')
print(f'cargo_2025:    {len(cargo_2025):,} rows')
print(f'cargo_yearly:  {len(cargo_yearly):,} rows')
print(f'top80:         {len(top80):,} rows')
print(f'counties:      {len(counties):,} rows')
print(f'amazon:        {len(amazon):,} rows')

---
## 1. National cargo overview

**Question:** What does the US cargo landscape look like, and how concentrated is it?

In [ ]:
# Concentration analysis
total_cargo_2025 = cargo_2025['total_throughput_lb'].sum()
cargo_sorted = cargo_2025.sort_values('total_throughput_lb', ascending=False).reset_index(drop=True)
cargo_sorted['cumshare'] = cargo_sorted['total_throughput_lb'].cumsum() / total_cargo_2025

for n in [3, 5, 10, 20, 50]:
    share = cargo_sorted.head(n)['total_throughput_lb'].sum() / total_cargo_2025
    print(f'Top {n:>3} airports handle {share:>5.1%} of US domestic cargo')

**Insight:** US air cargo is extremely concentrated. The top 3 airports alone handle a third of national volume, and the top 20 capture nearly all of it. This shape matters because **most of the 494 candidate airports have effectively zero cargo activity today** — which doesn't disqualify them as future hubs, but it tells us that Amazon's next hub will almost certainly be a deliberate strategic plant, not an opportunistic expansion of existing flow.

In [ ]:
# Top 20 cargo airports — bar chart with Amazon Air flag
fig, ax = plt.subplots(figsize=(11, 7))

top20 = top80.head(20).copy()
top20['throughput_M_lbs'] = top20['total_throughput_lb'] / 1e6
top20 = top20.sort_values('throughput_M_lbs')  # for horizontal bar order

colors = [AMAZON_ORANGE if v else NEUTRAL for v in top20['is_amazon_air']]
ax.barh(top20['ident'], top20['throughput_M_lbs'], color=colors)

ax.set_xlabel('2025 Cargo Throughput (millions of lbs)')
ax.set_ylabel('')
ax.set_title('Top 20 US Cargo Airports — 2025\nOrange = Amazon Air presence, Gray = no Amazon Air',
             loc='left', fontsize=13, pad=14)
ax.grid(axis='x', alpha=0.3)

for i, (_, row) in enumerate(top20.iterrows()):
    ax.text(row['throughput_M_lbs'] + 50, i, f"  {row['throughput_M_lbs']:,.0f}",
            va='center', fontsize=9, color=NEUTRAL)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'top20_cargo_airports.png', bbox_inches='tight')
plt.show()

---
## 2. Year-over-year cargo trends (2020–2025)

**Question:** Is US cargo growing, flat, or declining? Are the top airports gaining or losing share?

In [ ]:
# National cargo by year
yearly_total = cargo_yearly.groupby('year')['total_throughput_lb'].sum() / 1e6  # M lbs

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(yearly_total.index, yearly_total.values, marker='o', linewidth=2.5,
        color=AMAZON_ORANGE, markersize=8)
ax.fill_between(yearly_total.index, yearly_total.values, alpha=0.12, color=AMAZON_ORANGE)

for x, y in yearly_total.items():
    ax.annotate(f'{y:,.0f}', (x, y), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=10, color=NEUTRAL)

ax.set_xlabel('Year')
ax.set_ylabel('Total US Cargo Throughput (M lbs)')
ax.set_title('US Domestic Cargo Throughput, 2020–2025', loc='left', fontsize=13, pad=12)
ax.set_ylim(0, yearly_total.max() * 1.15)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'cargo_trend_2020_2025.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top 5 airports — share over time
top5_ids = top80.head(5)['ident'].tolist()
trend = (
    cargo_yearly[cargo_yearly['airport_id'].isin(top5_ids)]
    .pivot_table(index='year', columns='airport_id', values='total_throughput_lb', aggfunc='sum')
    / 1e6
)
trend = trend[top5_ids]  # column order = ranking order

fig, ax = plt.subplots(figsize=(11, 6))
palette = sns.color_palette('rocket', n_colors=len(top5_ids))
for i, col in enumerate(trend.columns):
    ax.plot(trend.index, trend[col], marker='o', linewidth=2,
            label=col, color=palette[i])
ax.set_xlabel('Year')
ax.set_ylabel('Cargo Throughput (M lbs)')
ax.set_title('Top 5 Cargo Airports — Year-over-Year Trajectory', loc='left', fontsize=13, pad=12)
ax.legend(title='Airport', loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'top5_yearly_trend.png', bbox_inches='tight')
plt.show()

**Observations:**
- Total cargo dipped 2022–2023 after pandemic-era highs (2020–2021 was peak e-commerce demand)
- 2024–2025 stabilized but hasn't recovered to peak — the post-pandemic correction is real
- Top airports' relative ranking has been remarkably stable: SDF, MEM, ORD, CVG hold the top spots throughout
- Implication for hub-location: rankings derived from 2025 are not anomalous — they reflect the steady state

---
## 3. Geographic distribution

**Question:** Where is cargo concentrated geographically? Are there underserved regions?

In [ ]:
# Cargo by state (aggregating across all airports in that state)
cargo_by_state = (
    top80.groupby('state')['total_throughput_lb'].sum().sort_values(ascending=False)
    / 1e6
)

fig, ax = plt.subplots(figsize=(11, 6))
top_states = cargo_by_state.head(15)
ax.bar(top_states.index, top_states.values, color=ACCENT)
ax.set_xlabel('State')
ax.set_ylabel('2025 Cargo (M lbs)')
ax.set_title('Top 15 States by Cargo Activity (sum across top-80 airports)', loc='left', fontsize=13, pad=12)
for i, v in enumerate(top_states.values):
    ax.text(i, v + max(top_states.values)*0.01, f'{v:,.0f}',
            ha='center', fontsize=9, color=NEUTRAL)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'cargo_by_state.png', bbox_inches='tight')
plt.show()

In [ ]:
# Geographic scatter — all 494 candidate airports, sized by cargo
fig, ax = plt.subplots(figsize=(13, 7))

# Plot all candidates as small dots
ax.scatter(candidates['lon'], candidates['lat'], s=6, color=MUTED, alpha=0.45,
           label='Candidate airports (494)')

# Overlay top-80 cargo airports, sized by throughput
top80_pos = top80[top80['lon'].notna()]
sizes = (top80_pos['total_throughput_lb'] / top80_pos['total_throughput_lb'].max()) * 400 + 20
ax.scatter(top80_pos['lon'], top80_pos['lat'], s=sizes,
           color=NEUTRAL, alpha=0.4, edgecolor='none', label='Top-80 cargo airports')

# Overlay Amazon Air facilities with orange markers
ax.scatter(amazon['longitude'], amazon['latitude'], s=80, color=AMAZON_ORANGE,
           marker='^', edgecolor='black', linewidth=0.5, label='Amazon Air facilities')

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('US Air-Cargo Landscape — All Candidates + Top-80 + Amazon Air Facilities',
             loc='left', fontsize=13, pad=12)
ax.set_xlim(-130, -65)
ax.set_ylim(23, 50)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'geographic_landscape.png', bbox_inches='tight')
plt.show()

**Observations:**
- Cargo concentrates in the Midwest (KY, TN, OH, IL — the integrator-hub belt) and on the coasts
- Amazon Air's existing footprint covers the Midwest hub belt and major coastal gateways well
- **Geographic gaps that stand out:** Mountain West (NV, UT, CO, NM minus PHX), Upper Midwest beyond MSP, Pacific Northwest beyond SEA
- The South Central US (TX has AFW + IAH + SAT + AUS) is the most aggressively covered region after the Midwest

---
## 4. Airport capacity vs. activity

**Question:** Where is there infrastructure underutilization — airports with strong runways but low cargo activity? Those are exactly the hub-conversion candidates.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# Compute throughput in M lbs (log scale to handle the wide range)
top80_plot = top80.copy()
top80_plot['throughput_M'] = top80_plot['total_throughput_lb'] / 1e6
top80_plot['throughput_M'] = top80_plot['throughput_M'].clip(lower=0.1)  # avoid log(0)

amazon_mask = top80_plot['is_amazon_air']
ax.scatter(top80_plot.loc[~amazon_mask, 'longest_runway_ft'],
           top80_plot.loc[~amazon_mask, 'throughput_M'],
           s=70, color=NEUTRAL, alpha=0.55, label='No Amazon Air presence')
ax.scatter(top80_plot.loc[amazon_mask, 'longest_runway_ft'],
           top80_plot.loc[amazon_mask, 'throughput_M'],
           s=70, color=AMAZON_ORANGE, alpha=0.9, label='Amazon Air present')

# Annotate top 12 by cargo for context
for _, row in top80_plot.head(12).iterrows():
    ax.annotate(row['ident'], (row['longest_runway_ft'], row['throughput_M']),
                textcoords='offset points', xytext=(6, 4), fontsize=9, color=NEUTRAL)

ax.set_xlabel('Longest Runway (ft)')
ax.set_ylabel('2025 Cargo Throughput (M lbs, log scale)')
ax.set_yscale('log')
ax.set_title('Airport Capacity vs. Current Activity (Top 80)', loc='left', fontsize=13, pad=12)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'capacity_vs_activity.png', bbox_inches='tight')
plt.show()

In [ ]:
# Runway length distribution across full 494 candidates
fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(candidates['longest_runway_ft'], bins=40, color=ACCENT, alpha=0.7, ax=ax)
ax.axvline(7000, color='red', linestyle='--', linewidth=1, alpha=0.6, label='7,000 ft (universe cutoff)')
ax.axvline(10000, color=AMAZON_ORANGE, linestyle='--', linewidth=1.5, label='10,000 ft (heavy-jet ideal)')
ax.set_xlabel('Longest Runway (ft)')
ax.set_ylabel('Number of Airports')
ax.set_title(f'Runway Length Distribution Across {len(candidates)} Candidate Airports',
             loc='left', fontsize=13, pad=12)
ax.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'runway_distribution.png', bbox_inches='tight')
plt.show()

**Observations:**
- Most candidate airports cluster at 7,000–9,000 ft — adequate for 767 ops but not abundant headroom
- A meaningful tail of airports has 10,000+ ft runways — these are the heavy-jet-ideal candidates
- The capacity-vs-activity scatter is the most strategically revealing chart: **points high on runway length but low on cargo are underutilized infrastructure** — exactly what we want to surface in scoring

---
## 5. Population catchment preview

**Question:** What does the US population look like at the county level, and where is the catchment opportunity?

In [ ]:
# County population summary stats
print(f'County count: {len(counties):,}')
print(f'Total US population (sum): {counties["population"].sum():,.0f}')
print(f'\nDistribution of county populations:')
print(counties['population'].describe(percentiles=[.5, .75, .9, .99]).to_string())
print(f'\nTop 10 most populous counties:')
print(counties.nlargest(10, 'population')[['county_name', 'population']].to_string(index=False))

In [ ]:
# Quick map of county population — log scale
fig, ax = plt.subplots(figsize=(13, 7))
counties_us = counties[
    (counties['centroid_lon'].between(-130, -65)) &
    (counties['centroid_lat'].between(23, 50))
].copy()

sc = ax.scatter(counties_us['centroid_lon'], counties_us['centroid_lat'],
                c=np.log10(counties_us['population'].clip(lower=1)),
                cmap='viridis', s=8, alpha=0.7)
cb = plt.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label('log10(population)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('US County Population (county centroids, log color scale)',
             loc='left', fontsize=13, pad=12)
ax.set_xlim(-130, -65); ax.set_ylim(23, 50)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'county_population_map.png', bbox_inches='tight')
plt.show()

**Observations:**
- US population concentrates along the coasts (BosWash, California, Pacific NW, FL, Texas Triangle, Front Range)
- The interior is sparse outside metro nodes (Chicago, Twin Cities, Denver, Salt Lake)
- This shape will dominate the catchment dimension: airports near population concentrations will win, airports in low-density regions will lose
- Catchment computation in notebook 03 will use 4-hour-drive equivalent radius (~240 miles)

---
## 6. Amazon Air coverage gap analysis

**Question:** Among the top cargo airports, which lack Amazon Air presence? Those are the strategic gaps the Hub Fitness Index will rank.

In [ ]:
# Top 30 — Amazon vs no-Amazon
top30 = top80.head(30).copy()
top30['throughput_M'] = top30['total_throughput_lb'] / 1e6

with_amazon = top30[top30['is_amazon_air']]
without_amazon = top30[~top30['is_amazon_air']]

print(f'Top 30 cargo airports — Amazon Air coverage:')
print(f'  With Amazon presence: {len(with_amazon)} airports')
print(f'  Without:              {len(without_amazon)} airports')
print(f'\nTop cargo airports WITHOUT Amazon Air (these are the strategic-gap candidates):')
gap_view = without_amazon[['cargo_rank_2025','ident','name','state','throughput_M']].rename(
    columns={'cargo_rank_2025':'rank','throughput_M':'M_lbs'})
print(gap_view.to_string(index=False))

In [ ]:
# Coverage gap visualization
fig, ax = plt.subplots(figsize=(11, 8))
top30_sorted = top30.sort_values('throughput_M')
colors = [AMAZON_ORANGE if v else NEUTRAL for v in top30_sorted['is_amazon_air']]
ax.barh(top30_sorted['ident'], top30_sorted['throughput_M'], color=colors)
ax.set_xlabel('2025 Cargo Throughput (M lbs)')
ax.set_title('Top 30 Cargo Airports — Amazon Air Coverage Gap\nOrange = Amazon present, Gray = strategic-gap candidate',
             loc='left', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'dashboards' / 'amazon_coverage_gap.png', bbox_inches='tight')
plt.show()

**Interpretation of the gap:**

Not every gap is a real opportunity. Some top-cargo airports lack Amazon Air for solid strategic reasons:

- **MEM** — FedEx home base; competing here would be a war, not an entry
- **ORD, ATL, JFK, LAX, DFW** — passenger-heavy major hubs; high slot constraints, expensive real estate
- **OAK, HNL** — geographic edge cases; OAK overlaps with SFO (Amazon-served), HNL is offshore

Other gaps are more interesting:
- **ONT** — Ontario, CA: high cargo, less congested than LAX, secondary metro Los Angeles
- **PHX** — large metro, central-Southwest gap in Amazon's footprint
- **OAK** — Oakland: alternative to SFO with more cargo-friendly infrastructure

**The Hub Fitness Index in notebook 03 will quantify which of these gaps actually scores high once we layer in catchment, weather, and complementarity.**

---
## 7. Key findings — heading into Hub Fitness Index

Five takeaways from this EDA that should inform scoring:

1. **Cargo is highly concentrated.** The top 20 airports handle the overwhelming majority of US air cargo. New-hub scoring should not just reward existing cargo volume — that would just rank-order the incumbents.

2. **Cargo rankings are stable across years.** 2025 is a good representative sample; multi-year averaging adds noise without changing the answer.

3. **The Midwest hub belt is saturated.** SDF/CVG/IND/ORD/RFD/CMH/MEM are all within a few hundred miles of each other. A new Amazon hub near here adds little incremental coverage.

4. **Capacity is widely distributed but activity is not.** Many candidate airports have strong runways and low current cargo — these are the conversion candidates the index needs to surface.

5. **Population catchment will dominate the scoring.** Coastal counties are dense, interior counties are sparse. Catchment differences between candidates will be 5× or 10×, while other dimensions vary by 2× at most.

## Next steps

- Pull NOAA weather data for the 80 airports identified in `top_cargo_airports.csv`
- Build `03_hub_scoring.ipynb` — construct the Hub Fitness Index with weighted scoring across all 5 dimensions
- Run sensitivity analysis on the weights
- Identify final top 3 recommended airports